In [1]:
%load_ext autoreload
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, project_root)

In [2]:
import pandas as pd

In [3]:
from src.pipeline.seasonality_etl import SeasonalityETL
from src.visualization.interactive_plots import show_interactive_seasonality_plot
from src.visualization.synthetic_data_generator import (
    generate_perfect_seasonality_all_intervals,
)

### Create data

In [4]:
df_synth = generate_perfect_seasonality_all_intervals(seed=42)

df_synth["ticker"] = "PERFECT.SYN"
df_synth["date"] = pd.to_datetime(df_synth["date"])

etl = SeasonalityETL()
df_rolling = etl.fit_rolling(
    df_synth,
    frequencies=["W", "ME", "QE", "YE"] 
)

### Basic checks

In [5]:
print(df_synth.shape)
df_synth.head()

(2151, 4)


,date,ticker,interval,close
0,2020-01-01,PERFECT.SYN,1d,100.049671
1,2020-01-02,PERFECT.SYN,1d,101.995835
2,2020-01-03,PERFECT.SYN,1d,103.174828
3,2020-01-04,PERFECT.SYN,1d,103.210337
4,2020-01-05,PERFECT.SYN,1d,102.418414


In [6]:
print(df_rolling.shape)
df_rolling.head()

(376, 10)


,ticker,interval,freq,window_start,acf_lag_val,p2m_val,stl_strength,seasonality_score_linear,seasonality_score_geom,seasonality_score_harmonic
0,PERFECT.SYN,1d,ME,2020-01-31,0.146672,0.022608,0.0,0.056427,0.0,0.000003
1,PERFECT.SYN,1d,ME,2020-02-29,0.585488,0.376159,0.0,0.320549,0.0,0.000003
2,PERFECT.SYN,1d,ME,2020-03-31,0.688679,0.611713,0.0,0.433464,0.0,0.000003
3,PERFECT.SYN,1d,ME,2020-04-30,0.866955,0.892385,0.0,0.586447,0.0,0.000003
4,PERFECT.SYN,1d,ME,2020-05-31,0.919583,0.969467,0.0,0.629683,0.0,0.000003


# Metrics in Dashboard
## STL Seasonality Strength

Quantifies how much of a time series’ variation is explained by its seasonal component, as extracted using STL (Seasonal-Trend decomposition using Loess). It is calculated as:

$$S = \max\left(0,\ 1 - \frac{\operatorname{Var}(\text{remainder})}{\operatorname{Var}(\text{remainder} + \text{seasonal})} \right)$$

Values near 1 indicate strong seasonality; near 0 means weak or no seasonality.

**STL Rule of Thumb:**
- Seasonal line should have regular up-down waves, consistent amplitude and visible periodicity.
- It should also oscillate around zer and not overlap much with trend.
- Trend should capture log-term drift.
- Score domain is $> 0.6$: strong seasonality, $0.3 - 0.6$: moderate seasonality, $< 0.3$: negligible seasonality.


## ACF at Seasonal Lag

Refers to the value of the autocorrelation function at the lag corresponding to the seasonality period (e.g., lag = 12 for monthly data with yearly seasonality).

$$\text{ACF}_{s} = \rho(s)$$

where:
- $\text{ACF}_{s}$ is the autocorrelation at seasonal lag $s$
- $\rho(s)$ denotes the autocorrelation function evaluated at lag $s$

**ACF Rule of Thumb**
- ACF at the expected lag is clearly elevated (local maxima good to have but not required).
- Spike stands out from general decay and monotonic decay with string persistence.
- Echo peaks at multiples of the lag are visible.
- Score domain is $> 0.5$: strong seasonality (albeit very rare), $0.3- 0.5$: moderate seasonality, $0.15-0.3$: weak, possibly noise induced, $< 0.15$: most probably not seasonal.

## Periodogram

The periodogram helps identify dominant periodicities in a time series by analyzing its spectral density. A high peak-to-background ratio suggests a strong recurring cycle (seasonality), while a flat spectrum indicates randomness.

$$\text{Peak-to-background ratio} = \frac{\max(P(f))}{\text{mean}(P(f))}$$

where:
- $P(f)$ is the power spectral density at frequency $fF$

**Periodogram Rule of Thumb**
- Sharp, narrow peak at non-zero frequency
- High peak-to-mean ratio ($>20–30$) and peak is not at frequency $≈ 0$
- PMR is only meaningful when paired with frequency analysis. A high PMR at frequency $≈ 0$ is likely just trend.
- Domain values might be misleading in terms of seasonality: $1-5$: no dominant frequency, $5-15$: possibly weak seasonality or multiple low peaks, $15-30$: seasonality could be present, check alignment, $>30$: check if peak is not 0, if so, strong seasonal signal, $>100$: Check peak, possibly trend dominated. 

## Seasonal subseries (ANOVA)

This approach visualizes seasonality by grouping data into seasonal periods (e.g., months) and comparing their distributions. If the between-group variance is high compared to within-group variance, it suggests strong seasonality. This is conceptually similar to a one-way ANOVA test.

$$F = \frac{\text{Between-group variance}}{\text{Within-group variance}}$$

where:
- Flat or similar boxplots across months → low seasonality
- Distinct differences across months → strong seasonality

**SSA Rule of Thumb**
- Clear, systematic differences across seasonal groups (e.g., months or weeks).
- Boxplots are well-separated in their medians, have relatively low overlap in interquartile ranges (IQRs)
- Outliers cluster differently per group
- Visual implication: Between-group variance ≫ within-group variance

In [7]:
# TODO: add aggregated metrics' explanations

In [8]:
show_interactive_seasonality_plot(df_rolling, df_synth)

interactive(children=(Dropdown(description='ticker', options=('PERFECT.SYN',), value='PERFECT.SYN'), Dropdown(…